# OFAC Sanctioned Vessel Tracker — Iran Program

Parses OFAC's SDN_ADVANCED.XML, extracts Iran-program-designated vessels, runs data quality
checks, and stages the result for a point-in-time join against Kpler Strait of Hormuz transit data.

**Schema note:** the advanced XML is fully relational, not flat attributes. This notebook's parser
was reverse-engineered and tested against a real 125MB pull (`sdn_advanced_2026-07-13.xml`,
2026-07-10 issue date). Key structure:

- IMO/MMSI live in a separate `<IDRegDocuments>` section (IDRegDocTypeID 1626 / 91264), linked by
  an internal `IdentityID` — not the entity's public `FixedRef`.
- Vessel type is not free text; it's a `DetailReferenceID` that must be resolved against a lookup
  table in `<ReferenceValueSets>`.
- Program tags (`IRAN-EO13902`, etc.) and the actual OFAC listing date live in `<SanctionsEntries>`,
  a section that comes *after* the vessel records in the file.
- The vessel's owning shell company is linked via `<ProfileRelationships>`
  (`RelationTypeID=15003`, "Owned or Controlled By").

Because `IDRegDocuments` comes before `DistinctParties` in the file, and `SanctionsEntries`/
`ProfileRelationships` come after, this parses in one streaming pass (`lxml.iterparse`) rather than
loading the whole 125MB tree into memory at once.


## 1. Setup

In [1]:
import re
import time
from collections import Counter, defaultdict
from datetime import date, datetime, timezone

import pandas as pd
from lxml import etree

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 80)


## 2. Config

If you're re-running this on a fresh pull, point `XML_PATH` at wherever you saved it. The endpoint
has moved before (OFAC migrated SLS hosts and namespaces in 2024), so a fetch cell is included but
commented out — uncomment and adjust if you want this notebook to also handle the download step,
otherwise just supply a path to a file you already have.


In [2]:
XML_PATH = "C:\\Users\\luiscarlos.gaitan\\OneDrive - Jain Global\\Coding\\vessel_crossing\\sdn_advanced_2026-07-13.xml"

SNAPSHOT_DATE = datetime.now(timezone.utc).date()

# Iran-specific program tags (unambiguous)
IRAN_SPECIFIC_PROGRAMS = {"IRAN-EO13902", "IRAN-EO13846", "IRAN", "IRAN-CON-ARMS-EO"}

# Broader tags that CAN indicate Iran-linked designations (IRGC-QF terrorism route, IFSR) but
# can also attach to non-Iran programs (e.g. Venezuela). Vessels with ONLY these tags and no
# Iran-specific tag get flagged for manual review rather than auto-included/excluded.
BROAD_IRAN_ADJACENT_PROGRAMS = {"IFSR", "SDGT", "NPWMD"}

VESSEL_PARTY_SUBTYPE_ID = "1"
IMO_DOC_TYPE = "1626"    # IDRegDocTypeID for "Vessel Registration Identification"
MMSI_DOC_TYPE = "91264"  # IDRegDocTypeID for "MMSI"

FEATURE_CALL_SIGN = "1"
FEATURE_VESSEL_TYPE = "2"
FEATURE_FLAG = "3"
FEATURE_OWNER = "4"
FEATURE_FORMER_FLAG = "24"
FEATURE_YEAR_BUILT = "967"

RELATION_OWNED_CONTROLLED = "15003"  # "Owned or Controlled By"


## 3. (Optional) Fetch

Uncomment to download fresh. Left inactive by default since you already have a pull to inspect.


In [3]:
# import requests
# SDN_ADVANCED_XML_URL = "https://sanctionslistservice.ofac.treas.gov/api/PublicationPreview/exports/SDN_ADVANCED.XML"
# HEADERS = {"User-Agent": "JainGlobal-EnergyResearch-SanctionsMonitor/1.0"}
#
# resp = requests.get(SDN_ADVANCED_XML_URL, headers=HEADERS, timeout=120)
# resp.raise_for_status()
# XML_PATH = f"sdn_advanced_{SNAPSHOT_DATE.isoformat()}.xml"
# with open(XML_PATH, "wb") as f:
#     f.write(resp.content)
# print(f"Downloaded {len(resp.content):,} bytes to {XML_PATH}")


## 4. Parse

Single streaming pass over the file. See the schema notes in the intro cell for why each section
is handled in this order.


In [4]:
def parse_sdn_advanced(xml_path: str) -> list:
    detail_reference_values = {}
    imo_by_identity_id = {}
    mmsi_by_identity_id = {}
    vessels = {}
    profile_names = {}

    context = etree.iterparse(xml_path, events=("end",), recover=True)

    for event, elem in context:
        tag = etree.QName(elem).localname

        if tag == "DetailReference":
            detail_reference_values[elem.get("ID")] = (elem.text or "").strip()
            elem.clear()
            continue

        if tag == "IDRegDocument":
            doc_type = elem.get("IDRegDocTypeID")
            identity_id = elem.get("IdentityID")
            reg_no_el = elem.find("{*}IDRegistrationNo")
            reg_no = (reg_no_el.text or "").strip() if reg_no_el is not None else None
            if reg_no:
                digits = re.sub(r"\D", "", reg_no)
                if digits and doc_type == IMO_DOC_TYPE:
                    imo_by_identity_id[identity_id] = digits
                elif digits and doc_type == MMSI_DOC_TYPE:
                    mmsi_by_identity_id[identity_id] = digits
            elem.clear()
            continue

        if tag == "DistinctParty":
            profile_el = elem.find(".//{*}Profile")
            if profile_el is not None:
                profile_id = profile_el.get("ID")
                fixed_ref = elem.get("FixedRef")

                identity_el = profile_el.find(".//{*}Identity")
                identity_id = identity_el.get("ID") if identity_el is not None else None
                name = None
                if identity_el is not None:
                    name_el = identity_el.find(".//{*}NamePartValue")
                    if name_el is not None:
                        name = name_el.text
                profile_names[profile_id] = name

                if profile_el.get("PartySubTypeID") == VESSEL_PARTY_SUBTYPE_ID:
                    features = {}
                    for feature in profile_el.findall(".//{*}Feature"):
                        ftype = feature.get("FeatureTypeID")
                        version = feature.find(".//{*}VersionDetail")
                        if version is None:
                            continue
                        detail_ref_id = version.get("DetailReferenceID")
                        if detail_ref_id:
                            features.setdefault(ftype, []).append(("ref", detail_ref_id))
                        elif version.text:
                            features.setdefault(ftype, []).append(("text", version.text))

                    vessels[profile_id] = {
                        "profile_id": profile_id,
                        "fixed_ref": fixed_ref,
                        "identity_id": identity_id,
                        "vessel_name": name,
                        "raw_features": features,
                        "linked_entity_profile_id": None,
                        "programs": [],
                        "first_listed_date": None,
                    }
            elem.clear()
            continue

        if tag == "ProfileRelationship":
            from_id = elem.get("From-ProfileID")
            to_id = elem.get("To-ProfileID")
            if elem.get("RelationTypeID") == RELATION_OWNED_CONTROLLED and from_id in vessels:
                vessels[from_id]["linked_entity_profile_id"] = to_id
            elem.clear()
            continue

        if tag == "SanctionsEntry":
            profile_id = elem.get("ProfileID")
            if profile_id in vessels:
                date_el = elem.find(".//{*}EntryEvent/{*}Date")
                if date_el is not None:
                    y, m, d = (date_el.find(f"{{*}}{p}") for p in ("Year", "Month", "Day"))
                    if y is not None and m is not None and d is not None:
                        try:
                            vessels[profile_id]["first_listed_date"] = (
                                f"{int(y.text):04d}-{int(m.text):02d}-{int(d.text):02d}"
                            )
                        except (ValueError, TypeError):
                            pass
                for measure in elem.findall(".//{*}SanctionsMeasure"):
                    comment_el = measure.find("{*}Comment")
                    if comment_el is not None and comment_el.text:
                        vessels[profile_id]["programs"].append(comment_el.text.strip())
            elem.clear()
            continue

    results = []
    for v in vessels.values():
        v["imo"] = imo_by_identity_id.get(v["identity_id"])
        v["mmsi"] = mmsi_by_identity_id.get(v["identity_id"])

        def resolve(ftype_id):
            out = []
            for kind, val in v["raw_features"].get(ftype_id, []):
                out.append(detail_reference_values.get(val, f"[unresolved:{val}]") if kind == "ref" else val)
            return out

        v["call_sign"] = next(iter(resolve(FEATURE_CALL_SIGN)), None)
        v["vessel_type"] = next(iter(resolve(FEATURE_VESSEL_TYPE)), None)
        v["flag"] = next(iter(resolve(FEATURE_FLAG)), None)
        v["former_flag"] = next(iter(resolve(FEATURE_FORMER_FLAG)), None)
        v["year_built"] = next(iter(resolve(FEATURE_YEAR_BUILT)), None)

        linked_id = v["linked_entity_profile_id"]
        v["linked_entity_name"] = profile_names.get(linked_id) if linked_id else None

        del v["raw_features"]
        del v["linked_entity_profile_id"]
        results.append(v)

    return results

t0 = time.time()
all_vessels = parse_sdn_advanced(XML_PATH)
print(f"Parsed {len(all_vessels)} total vessel-type records in {time.time()-t0:.1f}s")


Parsed 1497 total vessel-type records in 5.6s


## 5. Classify Iran relevance

Three buckets, not two — the middle one needs a human glance:
1. **Iran-specific tag present** → confidently Iran-relevant.
2. **Only broad tags** (SDGT/IFSR/NPWMD) with no Iran-specific tag → often still Iran-relevant
   (IRGC-Qods Force shipping fronts get designated under terrorism authorities, not just the
   petroleum EO) but occasionally belongs to an unrelated program (Venezuela, DPRK) that happens to
   share a tag. Use `linked_entity_name` to eyeball these.
3. **No Iran-adjacent tag at all** → excluded.


In [5]:
def classify(v):
    progs = set(v["programs"])
    if progs & IRAN_SPECIFIC_PROGRAMS:
        return "iran_specific"
    if progs & BROAD_IRAN_ADJACENT_PROGRAMS:
        return "broad_tag_only_REVIEW"
    return "not_iran"

for v in all_vessels:
    v["iran_classification"] = classify(v)

df_all = pd.DataFrame(all_vessels)
print(df_all["iran_classification"].value_counts())


iran_classification
not_iran                 780
iran_specific            607
broad_tag_only_REVIEW    110
Name: count, dtype: int64


In [6]:
df_all.to_csv(f"ofac_sdn_vessels_{SNAPSHOT_DATE.isoformat()}.csv", index=False)

In [7]:
# The review bucket - eyeball these before deciding to include/exclude
review_df = df_all[df_all["iran_classification"] == "broad_tag_only_REVIEW"][
    ["vessel_name", "imo", "programs", "linked_entity_name", "flag"]
]
review_df


,vessel_name,imo,programs,linked_entity_name,flag
341,Sarak,9226968,[SDGT],MEHDI GROUP,Iran
342,Sobar,9221970,[SDGT],MEHDI GROUP,Iran
343,Tour 2,9364112,[SDGT],KHADIJA SHIP MANAGEMENT PRIVATE LIMITED,Panama
344,Adrian Darya 1,9116412,[SDGT],None,Iran
358,Genava 12,9776523,"[SDGT, IFSR]",Khedri Jahan Darya Co,Iran
...,...,...,...,...,...
1315,SHRIA,9179347,[SDGT],None,Antigua and Barbuda
1316,BLACK ROCK,9196448,[SDGT],None,Panama
1394,Albarraq Z,9252943,[SDGT],Albarraq Shipping Co,Unknown
1423,LARA,9221475,[SDGT],None,St. Kitts & Nevis


Adjust `INCLUDE_REVIEW_BUCKET` after eyeballing the table above. Default is `True` since most of
these turned out to be genuine NIOC/IRGC-linked vessels in the test pull — but re-check each time
you refresh, since the composition can change.

In [8]:
INCLUDE_REVIEW_BUCKET = True

keep_classes = {"iran_specific"} | ({"broad_tag_only_REVIEW"} if INCLUDE_REVIEW_BUCKET else set())
df = df_all[df_all["iran_classification"].isin(keep_classes)].copy()
df["snapshot_date"] = SNAPSHOT_DATE
print(f"Final Iran-relevant vessel set: {len(df)}")
df.head(10)


Final Iran-relevant vessel set: 717


,profile_id,fixed_ref,identity_id,vessel_name,programs,first_listed_date,imo,mmsi,call_sign,vessel_type,flag,former_flag,year_built,linked_entity_name,iran_classification,snapshot_date
3,15036,15036,6780,ARTAVIL,[IRAN],2012-07-12,9187629,572469210,T2EU4,Crude/Oil Products Tanker,Iran,Malta,None,NATIONAL IRANIAN TANKER COMPANY,iran_specific,2026-07-29
4,15037,15037,6781,ARK III,[IRAN],2012-07-12,9187655,None,None,Crude/Oil Products Tanker,Iran,None,None,NATIONAL IRANIAN TANKER COMPANY,iran_specific,2026-07-29
5,15038,15038,6782,ARGO I,[IRAN],2012-07-12,9187667,None,T2EM4,Crude/Oil Products Tanker,Iran,Malta,None,NATIONAL IRANIAN TANKER COMPANY,iran_specific,2026-07-29
6,15039,15039,6783,ARNICA,[IRAN],2012-07-12,9187643,572467210,T2ES4,Crude/Oil Products Tanker,Iran,Malta,None,NATIONAL IRANIAN TANKER COMPANY,iran_specific,2026-07-29
7,15040,15040,6784,APAMA,[IRAN],2012-07-12,9187631,256845000,9HDS9,Crude/Oil Products Tanker,Iran,Tuvalu,None,NATIONAL IRANIAN TANKER COMPANY,iran_specific,2026-07-29
8,15041,15041,6785,BANEH,[IRAN],2012-07-12,8508462,422141000,EQKF,Landing Craft,Iran,None,None,NATIONAL IRANIAN TANKER COMPANY,iran_specific,2026-07-29
9,15042,15042,6786,DIAMOND II,[IRAN],2012-07-12,9218478,256865000,9HEG9,Crude Oil Tanker,Iran,Malta,None,NATIONAL IRANIAN TANKER COMPANY,iran_specific,2026-07-29
10,15043,15043,6787,DREAM II,[IRAN],2012-07-12,9356593,677049200,5IM 592,Crude Oil Tanker,Iran,Cyprus,None,NATIONAL IRANIAN TANKER COMPANY,iran_specific,2026-07-29
11,15044,15044,6788,DEEP SEA,[IRAN],2012-07-12,9218492,256862000,9HEE9,Crude Oil Tanker,Iran,Malta,None,NATIONAL IRANIAN TANKER COMPANY,iran_specific,2026-07-29
12,15045,15045,6789,DORE,[IRAN],2012-07-12,9357717,677049300,5IM 593,Crude Oil Tanker,Iran,Cyprus,None,NATIONAL IRANIAN TANKER COMPANY,iran_specific,2026-07-29


## 6. Data quality checks

1. Missing IMO (can't join to Kpler without one — track separately, don't silently drop)
2. IMO check-digit validation (catches parsing errors independent of OFAC's own data quality)
3. Duplicate IMO across different `profile_id`s (re-designation / relisting under a new entity)
4. Distributions: program tag, vessel type, flag state


In [9]:
missing_imo = df[df["imo"].isna()]
print(f"Missing IMO: {len(missing_imo)} of {len(df)}")
missing_imo[["vessel_name", "mmsi", "programs"]]


Missing IMO: 0 of 717


,vessel_name,mmsi,programs


In [10]:
def imo_checksum_valid(imo):
    digits = re.sub(r"\D", "", str(imo))
    if len(digits) != 7:
        return False
    weights = [7, 6, 5, 4, 3, 2]
    total = sum(int(d) * w for d, w in zip(digits[:6], weights))
    return total % 10 == int(digits[6])

df_with_imo = df[df["imo"].notna()].copy()
df_with_imo["imo_checksum_valid"] = df_with_imo["imo"].apply(imo_checksum_valid)
invalid = df_with_imo[~df_with_imo["imo_checksum_valid"]]
print(f"IMOs failing checksum: {len(invalid)} of {len(df_with_imo)}")
invalid[["vessel_name", "imo", "programs"]]


IMOs failing checksum: 0 of 717


,vessel_name,imo,programs


In [11]:
dupe_imos = df_with_imo[df_with_imo.duplicated("imo", keep=False)].sort_values("imo")
print(f"IMOs appearing under more than one profile_id: {dupe_imos['imo'].nunique()}")
dupe_imos[["imo", "vessel_name", "profile_id", "first_listed_date", "programs"]]


IMOs appearing under more than one profile_id: 0


,imo,vessel_name,profile_id,first_listed_date,programs


In [12]:
print("--- Program tag counts ---")
print(pd.Series([p for progs in df["programs"] for p in progs]).value_counts())

print("\n--- Vessel type counts ---")
print(df["vessel_type"].value_counts(dropna=False))

print("\n--- Flag state counts (top 15) ---")
print(df["flag"].value_counts(dropna=False).head(15))

print("\n--- Listing date range ---")
print(df["first_listed_date"].dropna().min(), "to", df["first_listed_date"].dropna().max())


--- Program tag counts ---
IRAN-EO13902         294
IRAN                 188
IFSR                 128
IRAN-EO13846         125
NPWMD                122
SDGT                 114
RUSSIA-EO14024         7
UKRAINE-EO13662        6
VENEZUELA-EO13850      1
IRAN-CON-ARMS-EO       1
Name: count, dtype: int64

--- Vessel type counts ---
vessel_type
Crude Oil Tanker             279
LPG Tanker                    80
Chemical/Products Tanker      63
Container Ship                51
Oil Products Tanker           42
Chemical/Oil Tanker           41
Bulk Carrier                  40
General Cargo                 35
Products Tanker               21
Crude/Oil Products Tanker     14
Bunkering Tanker               8
Tug                            8
Passenger                      8
Asphalt/Bitumen Tanker         7
None                           5
Landing Craft                  2
Crew/Supply Vessel             2
Shuttle Tanker                 2
Platform Supply Ship           2
Offshore Tug/Supply Ship      

In [13]:
print("--- Linked owning entity, top 15 (shadow fleet operators concentration) ---")
print(df["linked_entity_name"].value_counts(dropna=False).head(15))


--- Linked owning entity, top 15 (shadow fleet operators concentration) ---
linked_entity_name
None                                              449
Islamic Republic of Iran Shipping Lines           119
NATIONAL IRANIAN TANKER COMPANY                    61
oceanlink maritime dmcc                            13
Hokoul SAL Offshore                                 4
KAI HENG LONG GLOBAL ENERGY LIMITED                 4
QATRAT ALNADA ALMASI SHIP MANAGEMENT L.L.C          4
Kurdos Shipping Inc.                                3
IMS Ltd                                             3
NIOC                                                2
KHADIJA SHIP MANAGEMENT PRIVATE LIMITED             2
RED SEA SHIP MANAGEMENT LLC                         2
Sai Saburi Consulting Services Private Limited      2
MEHDI GROUP                                         2
Trade Bridge Global Inc.                            2
Name: count, dtype: int64


## 7. Save snapshot

Append-only, keyed by `snapshot_date`. This feeds the point-in-time spell derivation once you have
multiple dated snapshots accumulated — a single pull only tells you "sanctioned as of this file's
issue date," not the full listing history.


In [14]:
snapshot_cols = ["profile_id", "fixed_ref", "vessel_name", "imo", "mmsi", "call_sign", "flag",
                  "former_flag", "vessel_type", "year_built", "programs", "first_listed_date",
                  "linked_entity_name", "iran_classification", "snapshot_date"]

out_df = df_with_imo[snapshot_cols].copy()
out_df["programs"] = out_df["programs"].apply(lambda p: ";".join(p))

try:
    snapshot_path = f"ofac_iran_vessels_snapshot_{SNAPSHOT_DATE.isoformat()}.parquet"
    out_df.to_parquet(snapshot_path, index=False)
    print(f"Saved {len(out_df)} records to {snapshot_path}")
except ImportError:
    # no pyarrow/fastparquet available - fall back to CSV rather than fail the run
    snapshot_path = f"ofac_iran_vessels_snapshot_{SNAPSHOT_DATE.isoformat()}.csv"
    out_df.to_csv(snapshot_path, index=False)
    print(f"pyarrow/fastparquet not available - saved as CSV instead: {snapshot_path}")

import os
history_path = "ofac_iran_vessels_history.csv"
out_df.to_csv(history_path, mode="a", header=not os.path.exists(history_path), index=False)


Saved 717 records to ofac_iran_vessels_snapshot_2026-07-29.parquet


## 8. Comparison scaffold — Kpler SoH transits

**Still need from you:** the actual shape of your Kpler SoH extract — is it already a Snowflake
table (name/columns?), a raw SDK/API pull, and is it per-vessel transit events or aggregated daily
flow? The join below assumes an IMO column and a transit-date column; swap the placeholder for the
real thing once confirmed.


In [15]:
# --- PLACEHOLDER: replace with your actual Kpler SoH extract ---
kpler_soh_transits = pd.DataFrame({
    "imo": out_df["imo"].head(3).tolist() + ["9999999"],
    "vessel_name": ["EXAMPLE"] * 4,
    "transit_date": pd.to_datetime(["2026-06-01", "2026-06-15", "2026-07-01", "2026-07-05"]),
    "direction": ["outbound", "outbound", "inbound", "outbound"],
    "cargo_type": ["crude", "crude", "products", "crude"],
})
# --- end placeholder ---

# Point-in-time-correct flag: was this IMO listed as of the transit date, using first_listed_date?
# (Full spell-based approach, handling delisting, needs multiple accumulated snapshots - see note
# above. This version is correct for "listed before transit and still in this snapshot," which is
# the common case since OFAC delistings of Iran shadow fleet vessels are relatively rare.)
lookup = out_df.dropna(subset=["first_listed_date"]).set_index("imo")["first_listed_date"]

def was_sanctioned_at_transit(row):
    listed_date = lookup.get(row["imo"])
    if listed_date is None:
        return False
    return pd.Timestamp(listed_date) <= row["transit_date"]

kpler_soh_transits["was_sanctioned_at_transit"] = kpler_soh_transits.apply(was_sanctioned_at_transit, axis=1)
kpler_soh_transits


,imo,vessel_name,transit_date,direction,cargo_type,was_sanctioned_at_transit
0,9187629,EXAMPLE,2026-06-01,outbound,crude,True
1,9187655,EXAMPLE,2026-06-15,outbound,crude,True
2,9187667,EXAMPLE,2026-07-01,inbound,products,True
3,9999999,EXAMPLE,2026-07-05,outbound,crude,False


## 9. Next steps

- Confirm the Kpler schema and wire section 8 to the real table
- Wire fetch + parse into `sources.yaml` as a `BaseScraper` subclass, loading into a
  `OFAC_SANCTIONED_VESSELS_SNAPSHOT` Snowflake table (append-only, one row per IMO per scrape date)
- Schedule via Dagster — Iran designations cluster around news events (as seen in the batches
  above), so weekly is a reasonable floor, but consider event-driven re-scrapes around known
  escalation dates
- Once 2+ snapshots exist, build the spell-derivation logic (SDN entries can be delisted, though
  it's uncommon for Iran shadow fleet vessels specifically) to make the point-in-time join fully
  robust rather than relying on `first_listed_date` alone
- Compute the rolling metric that answers the actual question: % of SoH-transiting tonnage that's
  unsanctioned over time, conditioned on Iranian-origin loadings — a spike there is the "clean
  fleet flush" signature, distinct from just more shadow-fleet activity generally
